In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

## Loading the data

In [2]:
df_productcodes = pd.read_csv("/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/BACI_HS17_V202601/product_codes_HS17_V202601.csv")
df_contrycodes = pd.read_csv("/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/BACI_HS17_V202601/country_codes_V202601.csv")
df_conversion_HS17_GSEC11 = pd.read_excel(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/other/HS6_to_GTAP11.xlsx",
    sheet_name="H5",
    header=1
    )
df_conversion_HS22_GSEC11 = pd.read_excel(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/other/HS6_to_GTAP11.xlsx",
    sheet_name="H6",
    header=1,
)

In [3]:
df_all = pd.DataFrame()
for data in tqdm(Path("/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/BACI_HS17_V202601").glob("BACI_*.csv")):
    df = pd.read_csv(data)
    df_all = pd.concat([df_all, df], ignore_index=True)

df_all = df_all.rename(columns={
    "t": "Year",
    "k": "HS6",
    "i": "Exporter",
    "j": "Importer",
    "v": "k$",
    "q": "Quantity",
})

8it [00:23,  2.94s/it]


### Merging the data in one df

In [4]:
df_merge_1 = pd.merge(
    df_all,
    df_contrycodes[["country_code", "country_iso3"]],
    left_on="Exporter",
    right_on="country_code",
    how="left",
)
df_merge_1 = (
    df_merge_1.drop(columns="Exporter")
    .rename(columns={"country_iso3": "Exporter"})
    .drop(columns="country_code")
)

In [5]:
df_merge_2 = pd.merge(
    df_merge_1,
    df_contrycodes[["country_code", "country_iso3"]],
    left_on="Importer",
    right_on="country_code",
    how="left",
)
df_merge_2 = (
    df_merge_2.drop(columns="Importer")
    .rename(columns={"country_iso3": "Importer"})
    .drop(columns="country_code")
)

In [6]:
df_merge_3 = pd.merge(
    df_merge_2,
    df_productcodes[["code", "description"]],
    left_on="HS6",
    right_on="code",
    how="left",
)
df_merge_3 = df_merge_3.drop(columns="HS6").rename(columns={"code": "HS6_code"})

In [7]:
df_merge_3

,Year,k$,Quantity,Exporter,Importer,HS6_code,description
0,2020,1.250,0.003,AFG,AND,710399,Stones: precious (other than diamonds) and sem...
1,2020,0.198,0.001,AFG,AGO,300431,"Medicaments: containing insulin, for therapeut..."
2,2020,1.287,0.007,AFG,AGO,840999,Engines: parts for internal combustion piston ...
3,2020,31.057,169.800,AFG,AZE,251512,"Marble and travertine: merely cut, by sawing o..."
4,2020,0.047,0.009,AFG,ARG,391000,Silicones: in primary forms
...,...,...,...,...,...,...,...
89207216,2019,25.426,1.218,ZMB,BFA,722830,"Steel, alloy: bars and rods, hot-rolled, hot-d..."
89207217,2019,21.926,1.000,ZMB,BFA,722880,"Steel, alloy or non-alloy: hollow drill bars a..."
89207218,2019,16.927,NaN,ZMB,BFA,820719,"Tools, interchangeable: rock drilling or earth..."
89207219,2019,320.518,59.400,ZMB,URY,240120,Tobacco: partly or wholly stemmed or stripped


In [8]:
df_conversion_HS17_GSEC11.head()

,Classification,Code,Description,Code Parent,GSEC3,GSEC3_rev
0,H5,10121,"Horses; live, pure-bred breeding animals",101,CTL,CTL
1,H5,10129,"Horses; live, other than pure-bred breeding an...",101,CTL,CTL
2,H5,10130,Asses; live,101,CTL,CTL
3,H5,10190,Mules and hinnies; live,101,CTL,CTL
4,H5,10221,"Cattle; live, pure-bred breeding animals",102,CTL,CTL


In [9]:
df_merge_4 = pd.merge(
    df_merge_3,
    df_conversion_HS17_GSEC11[["Code", "GSEC3_rev"]],
    left_on="HS6_code",
    right_on="Code",
    how="left",
)
df_merge_4 = df_merge_4.drop(columns="Code").rename(columns={"GSEC3_rev": "GSEC3_code"})

In [10]:
df_merge_4.head()

,Year,k$,Quantity,Exporter,Importer,HS6_code,description,GSEC3_code
0,2020,1.250,0.003,AFG,AND,710399,Stones: precious (other than diamonds) and sem...,OMF
1,2020,0.198,0.001,AFG,AGO,300431,"Medicaments: containing insulin, for therapeut...",BPH
2,2020,1.287,0.007,AFG,AGO,840999,Engines: parts for internal combustion piston ...,MVH
3,2020,31.057,169.800,AFG,AZE,251512,"Marble and travertine: merely cut, by sawing o...",oxt
4,2020,0.047,0.009,AFG,ARG,391000,Silicones: in primary forms,CHM


In [11]:
df_clean = df_merge_4.groupby(["Year", "Exporter", "Importer", "GSEC3_code"]).agg({"k$": "sum", "Quantity": "sum"}).reset_index()

In [12]:
df_clean.head(20)

,Year,Exporter,Importer,GSEC3_code,k$,Quantity
0,2017,ABW,AUT,EEQ,0.160,0.001
1,2017,ABW,AUT,ELE,64.057,0.007
2,2017,ABW,AUT,MVH,0.285,0.004
3,2017,ABW,AUT,OMF,44.876,0.046
4,2017,ABW,BEL,EEQ,7.547,0.060
5,2017,ABW,BEL,ELE,9.000,0.000
6,2017,ABW,BEL,I_S,5.596,49.580
7,2017,ABW,BEL,OMF,6.896,0.002
8,2017,ABW,BGR,OMF,1.863,0.014
9,2017,ABW,BHR,OCR,0.324,0.530


In [13]:
df_clean.to_parquet(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/GTAP11_grouped_data.parquet", index=False
)

In [14]:
# df_merge_4.to_parquet(
#     "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/BACI_HS17_V202601/df_merge.parquet",
#     index=False,
# )

## Using the full data

In [45]:
import pandas as pd

df_clean = pd.read_parquet(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/GTAP11_grouped_data.parquet"
)

In [46]:
df_clean.head()

,Year,Exporter,Importer,GSEC3_code,k$,Quantity
0,2017,ABW,AUT,EEQ,0.160,0.001
1,2017,ABW,AUT,ELE,64.057,0.007
2,2017,ABW,AUT,MVH,0.285,0.004
3,2017,ABW,AUT,OMF,44.876,0.046
4,2017,ABW,BEL,EEQ,7.547,0.060


In [47]:
EU27 = [
    "AUT",
    "BEL",
    "BGR",
    "HRV",
    "CYP",
    "CZE",
    "DNK",
    "EST",
    "FIN",
    "FRA",
    "DEU",
    "GRC",
    "HUN",
    "IRL",
    "ITA",
    "LVA",
    "LTU",
    "LUX",
    "MLT",
    "NLD",
    "POL",
    "PRT",
    "ROU",
    "SVK",
    "SVN",
    "ESP",
    "SWE",
]

eu_map = {code: "UE27" for code in EU27}

df_UE = df_clean.copy().replace({"Exporter": eu_map, "Importer": eu_map})


In [48]:
df_UE_clean = df_UE[(df_UE["Exporter"] == "UE27") & (df_UE["Importer"] == "CHN") | (df_UE["Exporter"] == "CHN") & (df_UE["Importer"] == "UE27")]

In [49]:
df_UE_clean.columns


Index(['Year', 'Exporter', 'Importer', 'GSEC3_code', 'k$', 'Quantity'], dtype='str')

In [50]:
exports_ue_chine = (
    df_UE_clean[(df_UE_clean["Exporter"] == "UE27") & (df_UE_clean["Importer"] == "CHN")]
    .groupby("GSEC3_code")["k$"]
    .sum()
)
imports_ue_chine = (
    df_UE_clean[(df_UE_clean["Exporter"] == "CHN") & (df_UE_clean["Importer"] == "UE27")]
    .groupby("GSEC3_code")["k$"]
    .sum()
)
df_bilan = pd.DataFrame(
    {
        "Exports_UE_vers_Chine": exports_ue_chine,
        "Imports_UE_depuis_Chine": imports_ue_chine,
    }
).fillna(0)

df_bilan["Solde_commercial"] = (
    df_bilan["Exports_UE_vers_Chine"] - df_bilan["Imports_UE_depuis_Chine"]
)
df_bilan = df_bilan.reset_index()


In [52]:
df_bilan

,GSEC3_code,Exports_UE_vers_Chine,Imports_UE_depuis_Chine,Solde_commercial
0,BPH,1.535093e+08,6.648059e+07,8.702868e+07
1,B_T,1.928527e+07,2.100638e+06,1.718463e+07
2,CHM,1.579529e+08,1.794961e+08,-2.154324e+07
3,CMT,1.875632e+06,4.438978e+06,-2.563347e+06
4,COA,8.123226e+04,4.114511e+05,-3.302188e+05
5,CTL,1.707833e+05,1.298391e+03,1.694849e+05
6,C_B,2.054021e+05,3.817410e+02,2.050204e+05
7,EEQ,1.470241e+08,5.897667e+08,-4.427426e+08
8,ELE,2.407716e+08,1.303935e+09,-1.063163e+09
9,ELY,0.000000e+00,5.620000e-01,-5.620000e-01
